# Capstone synthesis — prediction error across four kinds of violated expectation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maierav/ai_oscp_neuro/blob/main/notebooks/capstone_synthesis.ipynb)

This notebook reproduces the **capstone figure** (`figures/capstone_synthesis.png`) from the
committed per-paradigm summary tables. It does **not** re-stream NWB — each error type is
computed in its own Result notebook; this capstone only puts the four resulting
prediction-error indices (and the cross-scale feature-oddball comparison) on comparable axes.

**Headline:** three error types (frequency, learned order, learned timing) carry a positive
prediction-error index whose hierarchical-bootstrap CI excludes zero; the sensorimotor
(motor-contingency) case is **null** in the released data. Indices use the project
DvI = (deviant − control)/(|deviant| + |control|) convention.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, matplotlib as mpl
from pathlib import Path
DATA = Path("..")/"data" if (Path("..")/"data").exists() else Path("data")
CAP   = pd.read_csv(DATA/"capstone_error_types.csv")
CROSS = pd.read_csv(DATA/"capstone_crossscale.csv")
print("Error-type table:"); print(CAP.to_string(index=False))
print("\nCross-scale table:"); print(CROSS.to_string(index=False))

In [ ]:
pcol={"Feature-oddball":"#1b4079","Sequence":"#2c6e49","Duration / timing":"#8f5b0a","Sensorimotor":"#7d3c98"}
fig=plt.figure(figsize=(14,5.2))
gs=fig.add_gridspec(1,2,width_ratios=[1.55,1.0],left=0.055,right=0.985,top=0.80,bottom=0.14,wspace=0.24)
axA=fig.add_subplot(gs[0,0]); order=["Feature-oddball","Sequence","Duration / timing","Sensorimotor"]
for y,pname in zip(np.arange(len(order))[::-1], order):
    r=CAP[CAP.paradigm==pname].iloc[0]; c=pcol[pname]; isnull=not (r["lo"]>0 or r["hi"]<0)
    axA.plot([r["lo"],r["hi"]],[y,y],color=c,lw=2.5,solid_capstyle="round",alpha=0.45 if isnull else 1.0)
    axA.plot(r["median"],y,"o",color=c,ms=10,mfc="white" if isnull else c,mew=1.8)
    axA.text(r["hi"]+0.02,y,f"{r['median']:+.2f}{' (null)' if isnull else ''}",va="center",fontsize=9.5,color=c,fontweight="bold")
    axA.text(-0.58,y+0.28,pname+(" *" if isnull else ""),va="center",fontsize=10,fontweight="bold")
    axA.text(-0.58,y-0.02,f"{r['expectation']} \u00b7 {int(r['n_sess'])} sessions \u00b7 {int(round(r['frac_pos']*100))}% cells +",va="center",fontsize=7,color="0.4")
axA.axvline(0,color="k",lw=0.8,ls=":"); axA.set_ylim(-0.6,3.6); axA.set_xlim(-0.6,0.65); axA.set_yticks([])
axA.set_xlabel("prediction-error index (bounded \u22121\u20261; DvI vs matched control, except duration = timing-PE vs sensory)")
axA.set_title("A \u00b7 Three error types positive (CI excludes 0); sensorimotor (*) null",loc="left",fontsize=8.3)
for sp in ["left","top","right"]: axA.spines[sp].set_visible(False)
axB=fig.add_subplot(gs[0,1]); tcol={"ecephys":"#1b4079","mesoscope":"#c0392b"}; tlab={"ecephys":"Neuropixels","mesoscope":"Mesoscope"}
for y,tech in zip([1,0],["ecephys","mesoscope"]):
    r=CROSS[CROSS.technique==tech].iloc[0]; c=tcol[tech]
    axB.plot([r["lo"],r["hi"]],[y,y],color=c,lw=2.5,solid_capstyle="round"); axB.plot(r["median"],y,"o",color=c,ms=10)
    axB.text(r["hi"]+0.015,y,f"+{r['median']:.2f}",va="center",fontsize=9.5,color=c,fontweight="bold")
    axB.text(r["median"],y+0.22,tlab[tech],ha="center",fontsize=10,fontweight="bold")
axB.axvline(0,color="k",lw=0.8,ls=":"); axB.set_ylim(-0.5,1.6); axB.set_xlim(0,0.5); axB.set_yticks([])
axB.set_xlabel("feature-oddball DvI (90\u00b0)"); axB.set_title("B \u00b7 Feature-oddball across scales",loc="left",fontsize=8.3)
for sp in ["left","top","right"]: axB.spines[sp].set_visible(False)
fig.suptitle("Capstone \u2014 three kinds of violated expectation carry a positive PE signal; sensorimotor is null",fontsize=9,y=0.95)
fig.savefig("../figures/capstone_synthesis.png",dpi=185,bbox_inches="tight"); plt.show()

**Reproducing the inputs.** Each row of `capstone_error_types.csv` is produced by its Result
notebook: feature-oddball \u2192 `oddball_confirmatory_ecephys.ipynb`; sequence \u2192
`sequence_mismatch_ecephys.ipynb`; duration \u2192 `duration_mismatch_ecephys.ipynb`; sensorimotor
\u2192 `sensorimotor_mismatch_ecephys.ipynb`. The cross-scale row (`capstone_crossscale.csv`) is
from Result 2. CIs are hierarchical bootstraps (resample sessions, then units).

> **Note on the sensorimotor deviant.** The sensorimotor row uses the orientation-90° deviant
> (the best-powered motor-mismatch type), whereas the *original* single-session capstone used the
> omission. Either way the multi-session result is null (see Result 3), so the choice is not
> load-bearing — but it is a post-hoc deviant selection and is flagged as such.

> **On comparability across rows.** Feature-oddball, sequence, and sensorimotor are DvIs
> (deviant vs a physically-matched control stimulus). Duration is a *timing-PE index* — the
> omission response at the expected time, normalized by the standard sensory response — because
> an omission has no physically-matched stimulus control. Both are bounded −1…+1, but they are
> not the identical construction; the axis is for reading **sign and consistency across error
> types**, not a strict magnitude ranking.